# RQ5 – Sensitivity to Evaluation Metrics

**Research Question:** How does the relative ranking of candidate models change when different evaluation metrics (MAE, RMSE, R²) are considered?

**Dataset:** [Kaggle Link](https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

import glob, os

# Auto-detect the dataset file anywhere in the Colab/Kaggle environment
_search_paths = [
    '/kaggle/input/marketing-and-product-performance-dataset/',
    '/content/sample_data/',
    '/content/',
    '/content/drive/MyDrive/',
    '.',
]
_extensions = ['*.csv', '*.xlsx', '*.xls']
DATA_PATH = None

for _dir in _search_paths:
    for _ext in _extensions:
        _matches = glob.glob(os.path.join(_dir, '**', _ext), recursive=True) + glob.glob(os.path.join(_dir, _ext))
        _matches = [m for m in _matches if 'marketing' in m.lower() or 'product' in m.lower() or 'performance' in m.lower()]
        if _matches:
            DATA_PATH = _matches[0]
            break
    if DATA_PATH:
        break

# Final fallback: pick any CSV/Excel in /content
if DATA_PATH is None:
    for _ext in _extensions:
        _all = glob.glob(f'/content/**/{_ext}', recursive=True) + glob.glob(f'./**/{_ext}', recursive=True)
        if _all:
            DATA_PATH = _all[0]
            break

if DATA_PATH:
    print(f'Dataset found: {DATA_PATH}')
else:
    raise FileNotFoundError(
        'Dataset not found. Please upload the CSV/Excel file to Colab via:\n'
        '  Files panel (left sidebar) → Upload, then re-run this cell.')
RANDOM_STATE = 42

Dataset found: /content/marketing_and_product_performance 2.csv


In [2]:
# ── Load & Preprocess ─────────────────────────────────────────────────────────
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

candidates = [c for c in df.columns if any(k in c.lower() for k in ['revenue','sales','sale','income','profit','amount'])]
TARGET = candidates[0] if candidates else df.select_dtypes(include=np.number).var().idxmax()

df_clean = df.dropna(thresh=len(df)*0.5, axis=1).copy()
X = df_clean.drop(columns=[TARGET]).copy()
y = df_clean[TARGET]

for col in X.select_dtypes(include='object').columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_imp = SimpleImputer(strategy='median').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imp)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=RANDOM_STATE)

In [3]:
# ── Train Models & Collect Metrics ────────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(random_state=RANDOM_STATE),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':           XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, verbosity=0),
    'SVM':               SVR(kernel='rbf'),
    'k-NN':              KNeighborsRegressor(n_neighbors=5)
}

records = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    records.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})

perf_df = pd.DataFrame(records)

# Assign ranks (1 = best)
rank_df = perf_df[['Model']].copy()
rank_df['Rank by MAE']  = perf_df['MAE'].rank().astype(int)   # lower = better
rank_df['Rank by RMSE'] = perf_df['RMSE'].rank().astype(int)  # lower = better
rank_df['Rank by R2']   = perf_df['R2'].rank(ascending=False).astype(int)  # higher = better
rank_df

,Model,Rank by MAE,Rank by RMSE,Rank by R2
0,Linear Regression,2,2,2
1,Decision Tree,6,6,6
2,Random Forest,3,3,3
3,XGBoost,4,4,4
4,SVM,1,1,1
5,k-NN,5,5,5


In [4]:
rank_df.to_csv('tables/RQ5_metric_sensitivity.csv', index=False)
print('Table saved.')

Table saved.


In [5]:
# ── Figure: Bump / Slope Chart ────────────────────────────────────────────────
metrics_order = ['Rank by MAE', 'Rank by RMSE', 'Rank by R2']
palette = ['#2E4057','#048A81','#54C6EB','#F4A261','#E76F51','#6D6875']

fig, ax = plt.subplots(figsize=(10, 7))

for i, (_, row) in enumerate(rank_df.iterrows()):
    ranks = [row[m] for m in metrics_order]
    ax.plot(range(3), ranks, marker='o', linewidth=2.5, markersize=10,
            color=palette[i % len(palette)], label=row['Model'])
    ax.annotate(row['Model'], (2, ranks[-1]), xytext=(2.05, ranks[-1]),
                fontsize=9, va='center', color=palette[i % len(palette)])

ax.set_xticks(range(3))
ax.set_xticklabels(['MAE (lower=better)', 'RMSE (lower=better)', 'R² (higher=better)'], fontsize=11)
ax.set_yticks(range(1, len(rank_df)+1))
ax.set_yticklabels([f'Rank {r}' for r in range(1, len(rank_df)+1)])
ax.invert_yaxis()
ax.set_ylabel('Rank (1 = Best)', fontsize=12)
ax.set_title('Figure 5. Model Ranking Variation Across Evaluation Metrics\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures/RQ5_metric_sensitivity.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


## Summary

**RQ5** shows how model rankings shift across MAE, RMSE, and R² metrics using a bump/slope chart. Results in `tables/RQ5_metric_sensitivity.csv` and `figures/RQ5_metric_sensitivity.pdf`.